In [1]:
!pip install bert-score -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 925.6 kB/s eta 0:00:000:00:01


In [2]:
!pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 11.7 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from bert_score import score

In [4]:

MODEL_PATH = "/kaggle/input/models/koushikikundu/mistral-hr/pytorch/default/1"
VALIDATION_FILE = "/kaggle/input/datasets/koushikikundu/validation-set/hr_policy_qa_validation.jsonl"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map={"": 0}
)

model.eval()

print("Model loaded successfully!")
print("Model device:", model.device)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

Model loaded successfully!
Model device: cuda:0


In [5]:
validation_data=[]
with open("/kaggle/input/datasets/koushikikundu/validation-set/hr_policy_qa_validation.jsonl", "r") as files:
    for file in files:
        validation_data.append(json.loads(file))
print(f"Loaded {len(validation_data)} validation questions")

Loaded 36 validation questions


In [6]:
flat_validation_data = [
    item
    for group in validation_data
    for item in group
]

print(len(flat_validation_data))

72


In [7]:
def generate_answer(question):

    messages = [
        {
            "role": "user",
            "content": f"""You are an HR policy assistant.

Answer the following question based on the HR policy you were trained on.

Question:
{question}

Give a clear and precise answer."""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [8]:
results=[]
for i, item in enumerate(flat_validation_data):

    question = item["question"]
    expected_answer = item["answer"]

    model_answer = generate_answer(question)

    results.append({
        "question": question,
        "expected_answer": expected_answer,
        "model_answer": model_answer
    })


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [9]:
references = [item["expected_answer"] for item in results]
predictions = [item["model_answer"] for item in results]

P, R, F1 = score(
    predictions,
    references,
    lang="en",
    device="cpu",
    batch_size=2,
    verbose=True
)

precision = P.mean().item()
recall = R.mean().item()
f1 = F1.mean().item()

print("\n===== Validation Results =====")
print(f"Questions tested : {len(results)}")
print(f"Precision        : {precision:.4f}")
print(f"Recall           : {recall:.4f}")
print(f"BERTScore F1     : {f1:.4f}")
print(f"F1 Percentage    : {f1 * 100:.2f}%")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/72 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/36 [00:00<?, ?it/s]

done in 24.62 seconds, 2.92 sentences/sec

===== Validation Results =====
Questions tested : 72
Precision        : 0.9271
Recall           : 0.8946
BERTScore F1     : 0.9101
F1 Percentage    : 91.01%
